# Medical Document Classifier - EfficientNet-B0
This notebook trains a classifier to distinguish between prescriptions, reports, and non-medical images.

In [1]:
# Cell 1: Install dependencies and imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import json
import time
import copy
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Cell 2: Load dataset and split into train/val/test
data_dir = 'dataset_augmented/'
input_size = 224
batch_size = 16

# Define transforms
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load full dataset
full_dataset = datasets.ImageFolder(data_dir)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Classes found: {class_names}")

# Split into 80% train, 10% val, 10% test
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

# Apply specific transforms to each split
class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

train_loader = DataLoader(SubsetWithTransform(train_dataset, train_transforms), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(SubsetWithTransform(val_dataset, val_test_transforms), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(SubsetWithTransform(test_dataset, val_test_transforms), batch_size=batch_size, shuffle=False)

print(f"Dataset split: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

Classes found: ['non_medical', 'prescription', 'report']
Dataset split: Train=576, Val=72, Test=72


In [3]:
# Cell 3: Define model (EfficientNet-B0)
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final classifier layer
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
# Cell 4: Training loop
num_epochs = 20
best_val_loss = float('inf')
early_stopping_patience = 5
no_improvement_epochs = 0

print("Starting Training...")
for epoch in range(num_epochs):
    # Each epoch has a training and validation phase
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader
            
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)
        
        print(f'Epoch {epoch}/{num_epochs-1} - {phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        
        if phase == 'val':
            scheduler.step(epoch_loss)
            
            if epoch_loss < best_val_loss:
                best_val_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), 'best_classifier.pth')
                no_improvement_epochs = 0
                print("Saved new best model weights.")
            else:
                no_improvement_epochs += 1
                
    if no_improvement_epochs >= early_stopping_patience:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

print("Training Complete")

In [ ]:
# Cell 5: Evaluation and metrics
# Load best weights
model.load_state_dict(torch.load('best_classifier.pth'))
model.eval()

all_preds = []
all_labels = []

print("Running Evaluation on Test Set...")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
report = classification_report(all_labels, all_preds, target_names=class_names)
conf_matrix = confusion_matrix(all_labels, all_preds)

print(f"Overall Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix:")
print(conf_matrix)

# Save to results.txt
with open('results.txt', 'w') as f:
    f.write(f"Overall Test Accuracy: {test_accuracy:.4f}\n")
    f.write("\nClassification Report:\n")
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(str(conf_matrix))

In [ ]:
# Cell 6: Save model info
model_info = {
    "class_names": class_names,
    "input_size": input_size,
    "model_architecture": "efficientnet_b0",
    "test_accuracy": float(test_accuracy),
    "total_params": total_params,
    "trainable_params": trainable_params
}

with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print("Model information saved to model_info.json")